<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the core scientific and visualization libraries needed for the notebook.

In [ ]:
!pip install numpy matplotlib openbb

The openbb_terminal package requires its own installation process. Follow the official OpenBB documentation to install it, as it has system-level dependencies that pip alone may not resolve.

## Imports and setup

numpy handles numerical arrays, matplotlib.pyplot and matplotlib.animation work together to build and animate the yield curve chart, and openbb provides a unified interface to pull Treasury yield data from FRED.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import matplotlib as mpl
from openbb import obb
mpl.rcParams["animation.embed_limit"] = 100
obb.user.preferences.output_type = "dataframe"
%matplotlib inline

The %matplotlib notebook magic at the top enables interactive, animated plots inside Jupyter. Without it, the animation would not render in the notebook and you would only see a static frame.

We define the nine Treasury maturities that form the x-axis of our yield curve, then pull nominal yields for each one going back to 1985. The "inverted" column flags every date where the 30-year yield fell below the 3-month yield.

In [ ]:
maturities = ["3m", "6m", "1y", "2y", "3y", "5y", "7y", "10y", "30y"]
data = (
    obb.fixedincome.government.treasury_rates(
        start_date="1985-01-01",
        provider="federal_reserve",
    )
    .dropna(how="all")
    .drop(columns=["month_1", "year_20"])
)
data.columns = maturities
data.index = pd.to_datetime(data.index)
data["inverted"] = data["30y"] < data["3m"]

This boolean column is what drives the color change in the animation. When the curve inverts, the line turns red, giving us an immediate visual signal of the condition that has preceded every US recession since the 1960s. Building this flag directly in the DataFrame keeps the animation function simple and fast.

## Build the chart layout

We create a blank figure and a single subplot, then configure the axis ranges, tick positions, and labels so the chart is ready to receive animated data.

In [ ]:
M = len(maturities)
n = len(data)

x = np.arange(M)
Y = data.loc[:, maturities].to_numpy(dtype=float) * 100.0
dates = data.index.strftime("%Y-%m-%d").to_numpy()
inv = data["inverted"].to_numpy(dtype=bool)

max_frames = 500
step = max(1, n // max_frames)
frames = np.arange(0, n, step, dtype=int)

Setting the y-axis to 20% may seem high, but Treasury yields reached the mid-teens in the mid-1980s after the Volcker-era rate hikes. A fixed axis range ensures every frame uses the same scale, so you can visually compare yield levels across decades without the chart rescaling and distorting your perception.

## Animate the yield curve over time

The init function clears the line and sets a default title, giving FuncAnimation a clean starting state before the first frame renders.

In [ ]:
def init():
    line.set_data([], [])
    title.set_text("U.S. Treasury Bond Yield Curve")
    return line, title

def update(fi):
    line.set_data(x, Y[fi])
    line.set_color("r" if inv[fi] else "y")
    title.set_text(f"U.S. Treasury Bond Yield Curve ({dates[fi]})")
    return line, title

This function is called once before the animation loop begins. Returning the line object tells matplotlib which artist to redraw, which is required when blit=True for performance.

The animate function is called once per frame. It reads that day's yields from the DataFrame, colors the line red if the curve is inverted and yellow otherwise, then updates the title with the current date.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

(line,) = ax.plot([], [], lw=2, animated=True)

ax.set_xlim(-0.25, M - 0.75)
ax.set_ylim(0, 20)
ax.set_xticks(x)
ax.set_xticklabels(maturities, rotation=0)
ax.set_yticks([2, 4, 6, 8, 10, 12, 14, 16, 18])
ax.set_xlabel("Time to maturity")
ax.set_ylabel("Yield (%)")

title = ax.text(
    0.5, 1.02, "", transform=ax.transAxes, ha="center", va="bottom", animated=True
)

ani = animation.FuncAnimation(
    fig,
    update,
    frames=frames,
    init_func=init,
    interval=100,
    blit=True,
    repeat=True,
)

html = ani.to_jshtml()
plt.close(fig)
HTML(html)

The red-yellow color toggle is the core insight of this visualization. When you watch the animation, you will see the curve flatten, invert to red, and then a recession follows shortly after. This pattern repeats consistently across nearly four decades of data, which is why professional fund managers treat inversion as one of their most reliable warning signals.

FuncAnimation ties everything together by looping through every trading day in the dataset. The interval of 5 milliseconds keeps the animation fast enough to watch decades of history in under a minute.

Setting blit=True tells matplotlib to redraw only the line and title rather than the entire figure on each frame, which dramatically improves rendering speed. With roughly 10,000 trading days in the dataset, this optimization is the difference between a smooth animation and a sluggish one.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.